# Phase 2-3: FGSM Attack

Generates adversarial examples using Fast Gradient Sign Method (FGSM).
Creates perturbed images at multiple epsilon levels.

In [ ]:
!pip install ultralytics -q
!pip install torch torchvision -q

import os
import sys
import yaml
import shutil
from pathlib import Path
from datetime import datetime

import torch
import torch.nn as nn
from torchvision import transforms
import numpy as np
from PIL import Image
from ultralytics import YOLO
from tqdm import tqdm
import matplotlib.pyplot as plt

%matplotlib inline

# mount drive if in colab
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')

print("Imports OK")
print(f"CWD: {os.getcwd()}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# paths - update for your setup
BASE_PATH = "/content/drive/MyDrive/Colab Notebooks/data"
COLAB_LOCAL = "/content"

CONFIG = {
    "weightsFile": os.path.join(BASE_PATH, "training/results from training/weights/best.pt"),
    "sourceData": os.path.join(BASE_PATH, "training"),
    "localWeights": os.path.join(COLAB_LOCAL, "weights/best.pt"),
    
    # output
    "outputBase": os.path.join(BASE_PATH, "adversarial/fgsm"),
    
    # attack params
    "epsilonValues": [0.045, 0.075, 0.105],
    "imgSize": 640,
    "batchSize": 8,
}

print("Config:")
print(f"  Weights: {CONFIG['weightsFile']}")
print(f"  Data: {CONFIG['sourceData']}")
print(f"  Output: {CONFIG['outputBase']}")

# check weights
if os.path.exists(CONFIG['weightsFile']):
    size = os.path.getsize(CONFIG['weightsFile']) / 1e6
    print(f"\nWeights found ({size:.1f} MB)")
else:
    print("\nWeights not found - check path")

# check data
if os.path.exists(CONFIG['sourceData']):
    print("Data found")
    for split in ['train', 'val', 'test']:
        splitPath = os.path.join(CONFIG['sourceData'], split, 'images')
        if os.path.exists(splitPath):
            nImgs = len([f for f in os.listdir(splitPath) if f.endswith(('.jpg', '.png'))])
            print(f"  {split}: {nImgs} images")
else:
    print("Data not found")

print(f"\nEpsilon values: {CONFIG['epsilonValues']}")

In [ ]:
# create data.yaml for YOLO
print("Setting up data.yaml...")
os.makedirs("/content/production", exist_ok=True)

dataYaml = {
    'path': CONFIG['sourceData'],
    'train': 'train/images',
    'val': 'val/images',
    'test': 'test/images',
    'nc': 1,
    'names': ['tank']
}

with open("/content/production/data.yaml", 'w') as f:
    yaml.dump(dataYaml, f)

print("Created /content/production/data.yaml")

In [ ]:
# copy weights locally for faster access
print("Copying weights...")

if os.path.exists(CONFIG['weightsFile']):
    os.makedirs(os.path.dirname(CONFIG['localWeights']), exist_ok=True)
    shutil.copy2(CONFIG['weightsFile'], CONFIG['localWeights'])
    
    size = os.path.getsize(CONFIG['localWeights']) / 1e6
    print(f"Copied ({size:.1f} MB)")
    print(f"Local: {CONFIG['localWeights']}")
    modelPath = CONFIG['localWeights']
else:
    print("Weights not found, using Drive path")
    modelPath = CONFIG['weightsFile']

In [ ]:
# load model
print("\nLoading model...")
try:
    model = YOLO(modelPath)
    model.model.eval()
    model.model = model.model.to(device)
    print("Model loaded")
    print(f"Model on: {next(model.model.parameters()).device}")
except Exception as e:
    print(f"Error: {e}")

In [ ]:
def computeFgsmPerturbation(imgTensor, model, epsilon):
    """Compute FGSM perturbation for an image.
    
    Args:
        imgTensor: input image [1, 3, H, W]
        model: YOLO model
        epsilon: attack strength
    
    Returns:
        perturbation tensor
    """
    imgDevice = imgTensor.device
    
    # clone and enable gradients
    imgTensor = imgTensor.clone().detach().requires_grad_(True)
    
    # forward pass
    try:
        outputs = model.model(imgTensor)
    except RuntimeError as e:
        if "should be the same" in str(e):
            model.model = model.model.to(imgDevice)
            outputs = model.model(imgTensor)
        else:
            raise e
    
    # extract objectness for loss
    if isinstance(outputs, tuple):
        outputs = outputs[0]
    
    # compute loss from objectness scores
    if len(outputs.shape) >= 3 and outputs.shape[-1] > 4:
        loss = outputs[..., 4].mean()  # objectness
    else:
        loss = outputs.mean()
    
    # backprop
    model.model.zero_grad()
    loss.backward()
    
    # create perturbation using gradient sign
    perturbation = epsilon * imgTensor.grad.data.sign()
    
    return perturbation.detach()


def applyFgsmAttack(imgTensor, perturbation):
    """Apply FGSM perturbation to image.
    
    Args:
        imgTensor: original image
        perturbation: perturbation to apply
    
    Returns:
        adversarial image tensor
    """
    adversarial = imgTensor + perturbation
    adversarial = torch.clamp(adversarial, 0, 1)
    return adversarial


print("FGSM functions defined")

In [ ]:
# test on one image
print("="*50)
print("TESTING FGSM")
print("="*50)

model.model = model.model.to(device)
print(f"Model on: {device}")

testImgDir = Path(CONFIG['sourceData']) / "test" / "images"
testImages = list(testImgDir.glob("*.jpg")) + list(testImgDir.glob("*.png"))

if testImages:
    testImgPath = testImages[0]
    print(f"Testing on: {testImgPath.name}")
    
    # load and preprocess
    img = Image.open(testImgPath).convert('RGB')
    origSize = img.size
    img = img.resize((640, 640), Image.LANCZOS)
    
    transform = transforms.ToTensor()
    imgTensor = transform(img).unsqueeze(0).to(device)
    print(f"Image tensor on: {imgTensor.device}")
    
    # test different epsilons
    testEpsilons = [0.045, 0.075, 0.105]
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    
    # original
    axes[0, 0].imshow(imgTensor.squeeze(0).permute(1, 2, 0).cpu())
    axes[0, 0].set_title('Original')
    axes[0, 0].axis('off')
    axes[1, 0].axis('off')
    
    for idx, epsilon in enumerate(testEpsilons, 1):
        perturbation = computeFgsmPerturbation(imgTensor, model, epsilon)
        advTensor = applyFgsmAttack(imgTensor, perturbation)
        
        # adversarial image
        axes[0, idx].imshow(advTensor.squeeze(0).permute(1, 2, 0).cpu())
        axes[0, idx].set_title(f'ε={epsilon}')
        axes[0, idx].axis('off')
        
        # perturbation (amplified)
        diff = (advTensor - imgTensor).abs() * 10
        axes[1, idx].imshow(diff.squeeze(0).permute(1, 2, 0).cpu())
        axes[1, idx].set_title('Perturbation (10x)')
        axes[1, idx].axis('off')
        
        print(f"ε={epsilon}: max perturbation = {perturbation.abs().max().item():.4f}")
    
    plt.suptitle('FGSM Attack Visualization', fontsize=16)
    plt.tight_layout()
    plt.show()
    
    print("\nTest successful!")
else:
    print(f"No test images found at {testImgDir}")

In [ ]:
def processSplit(splitName, epsilon, transform):
    """Process all images in a split with FGSM.
    
    Args:
        splitName: 'train', 'val', or 'test'
        epsilon: attack strength
        transform: image transform
    
    Returns:
        number of successfully processed images
    """
    srcImgDir = Path(CONFIG['sourceData']) / splitName / "images"
    srcLblDir = Path(CONFIG['sourceData']) / splitName / "labels"
    
    # output directories
    epsStr = f"eps_{epsilon:.3f}".replace(".", "_")
    outDir = Path(CONFIG['outputBase']) / epsStr / splitName
    outImgDir = outDir / "images"
    outLblDir = outDir / "labels"
    
    outImgDir.mkdir(parents=True, exist_ok=True)
    outLblDir.mkdir(parents=True, exist_ok=True)
    
    # get images
    imgFiles = list(srcImgDir.glob("*.jpg")) + list(srcImgDir.glob("*.png"))
    
    if not imgFiles:
        print(f"  No images in {splitName}")
        return 0
    
    print(f"  {splitName}: {len(imgFiles)} images")
    
    successful = 0
    for imgPath in tqdm(imgFiles, desc=f"    {splitName}"):
        try:
            # load and preprocess
            img = Image.open(imgPath).convert('RGB')
            origSize = img.size
            imgResized = img.resize((640, 640), Image.LANCZOS)
            
            imgTensor = transform(imgResized).unsqueeze(0).to(device)
            
            # apply FGSM
            perturbation = computeFgsmPerturbation(imgTensor, model, epsilon)
            advTensor = applyFgsmAttack(imgTensor, perturbation)
            
            # save adversarial image
            advImg = transforms.ToPILImage()(advTensor.squeeze(0).cpu())
            advImg = advImg.resize(origSize, Image.LANCZOS)
            advImg.save(outImgDir / imgPath.name, quality=95)
            
            # save label with metadata
            lblPath = srcLblDir / (imgPath.stem + '.txt')
            outLblPath = outLblDir / (imgPath.stem + '.txt')
            
            with open(outLblPath, 'w') as f:
                f.write(f"# FGSM_ADVERSARIAL\n")
                f.write(f"# epsilon: {epsilon}\n")
                f.write(f"# generated: {datetime.now().isoformat()}\n")
                if lblPath.exists():
                    with open(lblPath) as orig:
                        f.write(orig.read())
            
            successful += 1
            
        except Exception as e:
            print(f"Error on {imgPath.name}: {e}")
    
    return successful

In [ ]:
# process all splits for all epsilon values
print("\n" + "="*60)
print("GENERATING ADVERSARIAL DATASETS")
print("="*60)
print(f"\nEpsilon values: {CONFIG['epsilonValues']}")
print(f"Splits: train, val, test")
print(f"Output: {CONFIG['outputBase']}")

transform = transforms.ToTensor()

totalProcessed = 0
stats = {}

for epsilon in CONFIG['epsilonValues']:
    print(f"\n" + "-"*50)
    print(f"Epsilon = {epsilon}")
    print("-"*50)
    
    epsTotal = 0
    epsStats = {}
    
    for split in ['train', 'val', 'test']:
        nProcessed = processSplit(split, epsilon, transform)
        epsTotal += nProcessed
        epsStats[split] = nProcessed
    
    stats[epsilon] = epsStats
    totalProcessed += epsTotal
    print(f"\n  Total for ε={epsilon}: {epsTotal}")

print("\n" + "="*60)
print("DONE")
print("="*60)
print(f"\nTotal adversarial images: {totalProcessed}")
print(f"Output: {CONFIG['outputBase']}")

In [ ]:
# verify generated datasets
print("="*50)
print("VERIFICATION")
print("="*50)

outBase = Path(CONFIG['outputBase'])
if outBase.exists():
    epsDirs = sorted(list(outBase.glob('eps_*')))
    print(f"\nGenerated {len(epsDirs)} epsilon directories:\n")
    
    for epsDir in epsDirs:
        epsVal = epsDir.name.replace('eps_', '').replace('_', '.')
        print(f"{epsDir.name} (ε={epsVal}):")
        
        totalImgs = 0
        totalLbls = 0
        
        for split in ['train', 'val', 'test']:
            imgDir = epsDir / split / 'images'
            lblDir = epsDir / split / 'labels'
            
            if imgDir.exists():
                nImgs = len(list(imgDir.glob('*.jpg')) + list(imgDir.glob('*.png')))
                nLbls = len(list(lblDir.glob('*.txt')))
                totalImgs += nImgs
                totalLbls += nLbls
                print(f"  {split:5}: {nImgs:4} images, {nLbls:4} labels")
        
        print(f"  Total: {totalImgs} images, {totalLbls} labels\n")
    
    # check a sample label
    print("Checking label metadata...")
    if epsDirs:
        sampleLblDir = epsDirs[0] / 'test' / 'labels'
        sampleLbls = list(sampleLblDir.glob('*.txt'))
        if sampleLbls:
            with open(sampleLbls[0]) as f:
                content = f.read()
            if 'FGSM_ADVERSARIAL' in content:
                print("FGSM metadata found")
                print("\nSample header:")
                for line in content.split('\n')[:5]:
                    print(f"  {line}")
else:
    print(f"Output not found: {outBase}")

In [ ]:
print("="*50)
print("SUMMARY")
print("="*50)

print("\nGenerated:")
print(f"  {len(CONFIG['epsilonValues'])} epsilon values: {CONFIG['epsilonValues']}")
print(f"  3 splits: train, val, test")
print(f"  Images perturbed with FGSM")
print(f"  Labels marked with metadata")

print("\nFor your defense analysis:")
print(f"  Detection threshold: 3σ = 0.126")
print(f"  Attacks below ε=0.126 may be hard to detect")
print(f"  Attacks above ε=0.126 should trigger detection")

print("\nNext steps:")
print("  1. Test YOLOv8 on adversarial images")
print("  2. Extract features from adversarial examples")
print("  3. Compare feature distances to clean baseline")
print("  4. Train anomaly detector")
print("  5. Evaluate defense at each epsilon")

print(f"\nOutput: {CONFIG['outputBase']}")